In [ ]:
# Hybrid Search Demo Notebook
# This notebook demonstrates the hybrid search feature that combines:
# 1. Term matching (BM25-style substring/fuzzy matching)
# 2. Hierarchy expansion (parent-child traversal)
# 3. Embedding similarity (semantic vector search)

In [ ]:
# Configure paths
MEDCAT_MODEL_PATH = (
    "/workspaces/snomed_methods/model_packs/medcat_model_pack_422d1d38fc58f158.zip"
)
SAPBERT_MODEL_PATH = (
    "/workspaces/snomed_methods/embedding_models/SapBERT-from-PubMedBERT-fulltext"
)
UK_CLINICAL_RF2_PATH = "/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z"

In [ ]:
# Import the HybridSearch module
import sys

sys.path.insert(0, "/workspaces/snomed_methods")

from hybrid_search import HybridSearch

In [ ]:
# Initialize HybridSearch with all available data sources
print("Initializing HybridSearch...")
hybrid_searcher = HybridSearch(
    uk_path=UK_CLINICAL_RF2_PATH,
    medcat_path=MEDCAT_MODEL_PATH,
    model_path=SAPBERT_MODEL_PATH,
    backend="transformers",
    device="cpu",
)
print("✓ HybridSearch initialized")

In [ ]:
# Basic search functionality
# Query: "meningioma"
print("Searching for concepts related to 'meningioma'...")

results = hybrid_searcher.search(
    query="meningioma",
    top_k=15,
    term_weight=0.3,
    hierarchy_weight=0.2,
    embedding_weight=0.5,
)

print(f"\n✓ Found {len(results)} results")
print(f"  Term matches: {results.term_matches}")
print(f"  Hierarchy matches: {results.hierarchy_matches}")
print(f"  Embedding matches: {results.embedding_matches}")

In [ ]:
# Display results in a clear table format
import pandas as pd


def display_results(results, top_n=15):
    """Display hybrid search results in a formatted table."""
    data = []
    for cui, term, score in results.results[:top_n]:
        scores = results.cui_scores.get(cui, {})
        term_score = scores.get("term", 0)
        hier_score = scores.get("hierarchy", 0)
        embed_score = scores.get("embedding", 0)

        data.append(
            {
                "CUI": cui,
                "Term Name": term,
                "Combined Score": round(score, 4),
                "Term Score": round(term_score, 4),
                "Hierarchy Score": round(hier_score, 4),
                "Embedding Score": round(embed_score, 4),
            }
        )

    df = pd.DataFrame(data)
    display(
        df.style.format(
            {
                "Combined Score": "{:.4f}",
                "Term Score": "{:.4f}",
                "Hierarchy Score": "{:.4f}",
                "Embedding Score": "{:.4f}",
            }
        )
    )


display_results(results)

In [ ]:
# Display detailed metrics showing contribution from each strategy
def display_result_metrics(results, top_n=15):
    """Display detailed contribution metrics for each search strategy."""
    print(f"\n{'='*80}")
    print("CONTRIBUTION METRICS BY SEARCH STRATEGY")
    print(f"{'='*80}\n")

    for i, (cui, term, combined_score) in enumerate(results.results[:top_n], 1):
        scores = results.cui_scores.get(cui, {})
        term_score = scores.get("term", 0)
        hier_score = scores.get("hierarchy", 0)
        embed_score = scores.get("embedding", 0)

        max_score = max(term_score, hier_score, embed_score)
        contrib = []
        if term_score == max_score and term_score > 0:
            contrib.append("Term")
        if hier_score == max_score and hier_score > 0:
            contrib.append("Hierarchy")
        if embed_score == max_score and embed_score > 0:
            contrib.append("Embedding")

        if not contrib:
            contrib.append("None")

        print(f"{i:2d}. CUI: {cui}")
        print(f"     Term: {term}")
        print(f"     Combined Score: {combined_score:.4f}")
        print(
            f"     Strategy Scores -> "
            f"Term: {term_score:.4f} | Hierarchy: {hier_score:.4f} | "
            f"Embedding: {embed_score:.4f}"
        )
        print(f"     Top Contributor: {', '.join(contrib)}")
        print()


display_result_metrics(results)

In [ ]:
# Demonstrate different weight configurations
# Query: "diabetes"

print("Testing different weight configurations for 'diabetes' query...\n")

# Configuration 1: Heavy term matching (term_weight=0.6)
print("=" * 80)
print("Configuration 1: Term-heavy (term_weight=0.6, hierarchy=0.2, embedding=0.2)")
print("=" * 80)

results_term_heavy = hybrid_searcher.search(
    query="diabetes",
    top_k=10,
    term_weight=0.6,
    hierarchy_weight=0.2,
    embedding_weight=0.2,
)

print("\nResults (first 5):")
for i, (cui, term, score) in enumerate(results_term_heavy.results[:5], 1):
    print(f"{i}. {term} (CUI: {cui}) - Combined Score: {score:.4f}")

In [ ]:
# Configuration 2: Heavy hierarchy expansion (hierarchy_weight=0.5)
print("=" * 80)
print(
    "Configuration 2: Hierarchy-heavy (term_weight=0.2, hierarchy=0.5, embedding=0.3)"
)
print("=" * 80)

results_hier_heavy = hybrid_searcher.search(
    query="diabetes",
    top_k=10,
    term_weight=0.2,
    hierarchy_weight=0.5,
    embedding_weight=0.3,
)

print("\nResults (first 5):")
for i, (cui, term, score) in enumerate(results_hier_heavy.results[:5], 1):
    print(f"{i}. {term} (CUI: {cui}) - Combined Score: {score:.4f}")

In [ ]:
# Configuration 3: Heavy embedding similarity (embedding_weight=0.7)
print("=" * 80)
print(
    "Configuration 3: Embedding-heavy (term_weight=0.1, hierarchy=0.2, embedding=0.7)"
)
print("=" * 80)

results_embed_heavy = hybrid_searcher.search(
    query="diabetes",
    top_k=10,
    term_weight=0.1,
    hierarchy_weight=0.2,
    embedding_weight=0.7,
)

print("\nResults (first 5):")
for i, (cui, term, score) in enumerate(results_embed_heavy.results[:5], 1):
    print(f"{i}. {term} (CUI: {cui}) - Combined Score: {score:.4f}")

In [ ]:
# Compare weight configurations side by side
def compare_configurations(results_list, config_names, top_n=5):
    """Compare results from different weight configurations."""
    print(f"\n{'='*80}")
    print("SIDE-BY-SIDE COMPARISON OF WEIGHT CONFIGURATIONS")
    print(f"{'='*80}\n")

    all_cuis = []
    for results in results_list:
        all_cuis.extend([c for c, _, _ in results.results[:top_n]])
    all_cuis = list(dict.fromkeys(all_cuis))  # Preserve order, remove duplicates

    print(f"{'Rank':<6} {'CUI':<15} {'Term':<30}", end="")
    for name in config_names:
        print(f" {name:>12}", end="")
    print("\n")

    for i, cui in enumerate(all_cuis[:top_n], 1):
        # Find term for this CUI (from first results)
        term = None
        for results in results_list:
            for c, t, _ in results.results:
                if c == cui:
                    term = t
                    break
            if term:
                break

        print(f"{i:<6} {cui:<15} {term[:28]:<30}", end="")
        for results in results_list:
            score = 0.0
            for c, t, s in results.results:
                if c == cui:
                    score = s
                    break
            print(f" {score:>12.4f}", end="")
        print()

In [ ]:
# Compare diabetes results with different weights
compare_configurations(
    [results_term_heavy, results_hier_heavy, results_embed_heavy],
    ["Term-Heavy", "Hier-Heavy", "Embed-Heavy"],
)

In [ ]:
# Test with another query: "hypertension"

print("Searching for concepts related to 'hypertension'...")

results_hypertension = hybrid_searcher.search(
    query="hypertension",
    top_k=15,
    term_weight=0.3,
    hierarchy_weight=0.2,
    embedding_weight=0.5,
)

print(f"\n✓ Found {len(results_hypertension)} results")
display_results(results_hypertension)

In [ ]:
# Display detailed metrics for hypertension
display_result_metrics(results_hypertension)